# Chapter 4 Questionnaire: Under the Hood - Training a Digit Classifier

## 1. How is a grayscale image represented on a computer? How about a color image?

A grayscale image is a 2D grid of numbers (a matrix). Each cell is a pixel with a value from 0 (white) to 255 (black). Values between represent shades of gray.

A color image is typically represented as 3 separate grids stacked together - one for Red, one for Green, one for Blue (RGB). Each pixel is actually 3 numbers. So a 28x28 color image would be a 3 x 28 x 28 tensor.

## 2. How are the files and folders in the MNIST_SAMPLE dataset structured? Why?

`MNIST_SAMPLE` has two top-level folders: `train/` and `valid/`. Inside each, there are subfolders named `3/` and `7/`, each containing PNG images of that digit.

The folder name IS the label - no file contains the label explicitly. This is a common convention for image datasets: use folder names as labels. The `train/valid` split follows standard ML practice: train on one set, evaluate on a separate set the model has never seen.

## 3. Explain how the "pixel similarity" approach to classifying digits works.

1. Stack all training images of 3s together, take the mean of each pixel position to get an "ideal 3" image. Do the same for 7s.
2. For any new image, compute its distance (L1 or L2 norm) to the ideal 3 and ideal 7.
3. Classify as whichever ideal image it's closer to.

This is just a statistical calculation - it's not real machine learning because there are no parameters to optimize and improve over time.

## 4. What is a list comprehension? Create one now that selects odd numbers from a list and doubles them.

A list comprehension is a concise Python syntax for creating a new list by applying an operation to each element of an existing iterable, optionally filtering. Syntax: `[expression for item in iterable if condition]`

It's faster and more readable than a traditional for-loop.

In [ ]:
nums = [1, 2, 3, 4, 5, 6, 7, 8, 9]
result = [n * 2 for n in nums if n % 2 != 0]
result  # [2, 6, 10, 14, 18]

## 5. What is a "rank-3 tensor"?

A tensor with 3 axes (dimensions). For MNIST images stacked together: rank-3 means shape `[N, 28, 28]` - N images, each 28 rows, 28 columns.

Rank = number of axes. You need 3 indices to locate a single element: `tensor[image_index, row, col]`.

## 6. What is the difference between tensor rank and shape? How do you get the rank from the shape?

- **Rank**: the number of axes/dimensions a tensor has
- **Shape**: the length (size) of each axis

`rank = len(tensor.shape)` or `tensor.ndim`

Example: tensor with shape `[6131, 28, 28]` has rank 3. Shape tells you the counts; rank tells you how many axes there are.

## 7. What are RMSE and L1 norm?

Both measure distance between two sets of numbers:

- **L1 norm** (Mean Absolute Error): take absolute difference of each pair, average them. `(a - b).abs().mean()`
- **RMSE** (Root Mean Squared Error / L2 norm): square the differences, average them, take square root. `((a - b)**2).mean().sqrt()`

RMSE penalizes large errors more heavily than L1 norm.

## 8. How can you apply a calculation on thousands of numbers at once, many thousands of times faster than a Python loop?

By using PyTorch tensors (or NumPy arrays) with broadcasting and element-wise operations. These run at C/CUDA speed under the hood instead of Python loop speed.

For example: `valid_3_tens - mean3` subtracts a single 28x28 image from 1010 images all at once, via broadcasting, with no Python loop.

## 9. Create a 3x3 tensor containing numbers 1 to 9. Double it. Select the bottom-right four numbers.

In [ ]:
t = tensor([[1,2,3],[4,5,6],[7,8,9]])
doubled = t * 2
doubled[1:, 1:]  # bottom-right 2x2: [[10,12],[14,18]]

## 10. What is broadcasting?

Broadcasting is PyTorch's ability to automatically expand a smaller tensor to match the shape of a larger tensor during operations, without actually copying data in memory.

Example: subtracting `mean3` (shape `[28,28]`) from `valid_3_tens` (shape `[1010,28,28]`) works because PyTorch pretends `mean3` is `[1010,28,28]` by repeating it across the first axis. No extra memory is used.

This lets you write clean, fast code without explicit loops.

## 11. Are metrics generally calculated using the training set or the validation set? Why?

The validation set. Because the training set has already been seen by the model - it could have memorized it without truly understanding. The validation set is unseen data, so metrics on it tell you whether the model actually generalizes (truly learns) or just overfits (memorizes).

## 12. What is SGD?

Stochastic Gradient Descent. It's the algorithm for updating model parameters (weights) to minimize the loss. For each mini-batch:
1. Compute predictions
2. Calculate loss
3. Calculate gradients (how much each parameter affects the loss)
4. Update parameters: `param = param - gradient * learning_rate`

"Stochastic" refers to using random mini-batches rather than the whole dataset.

## 13. Why does SGD use mini-batches?

Two reasons:
1. **Speed**: computing gradients on the whole dataset is too slow per step. Computing on a single item gives noisy, unstable gradients. Mini-batches balance accuracy and speed.
2. **GPU efficiency**: GPUs work best when processing many items in parallel. Mini-batches keep the GPU busy.

## 14. What are the seven steps in SGD for machine learning?

1. **Initialize** the weights (randomly)
2. **Predict** using current weights
3. Calculate the **loss** (how bad the predictions are)
4. Calculate the **gradient** (direction to change each weight)
5. **Step** the weights (update them using gradient * learning rate)
6. **Repeat** from step 2
7. **Stop** when accuracy stops improving or time runs out

## 15. How do we initialize the weights in a model?

Randomly. This may seem surprising, but because SGD automatically improves weights through gradient descent, the starting point doesn't matter much - the model will converge regardless. Random initialization with small values is the standard approach.

## 16. What is "loss"?

Loss is a number that measures how bad the model's predictions are. Lower loss = better model. It's the function that SGD tries to minimize by adjusting parameters.

For regression: often Mean Squared Error. For binary classification: often cross-entropy or the custom `mnist_loss` we built.

## 17. Why can't we always use a high learning rate?

A learning rate that's too high can:
- Cause the loss to get **worse** (overshooting the minimum)
- Make the model diverge instead of converge
- Bounce around the minimum without ever settling

A learning rate that's too low just takes too many steps. The right LR is a balance.

## 18. What is a "gradient"?

The gradient is the derivative of the loss with respect to each parameter. It tells you: "if I increase this parameter by a tiny amount, how much does the loss change?"

It gives both direction (increase or decrease) and magnitude (how much to change). PyTorch computes gradients automatically via `loss.backward()`.

## 19. Do you need to know how to calculate gradients yourself?

No. PyTorch calculates them automatically via automatic differentiation (autograd). You just need to understand what a gradient is (the slope - how much the loss changes when you tweak a parameter), not how to compute derivatives by hand.

## 20. Why can't we use accuracy as a loss function?

Accuracy is "all or nothing" - an image is either classified correctly or not. Tweaking a weight by a tiny amount almost never flips any prediction from correct to incorrect. So the gradient is 0 almost everywhere.

If gradient = 0, SGD has no signal to follow - the model can't learn. Loss functions must be **smooth** so small parameter changes produce small, meaningful changes in loss.

## 21. Draw the sigmoid function. What is special about its shape?

The sigmoid function takes any input (negative or positive, any magnitude) and squeezes it into the range (0, 1). It's smooth, always increasing, and has no jumps or flat spots.

`sigmoid(x) = 1 / (1 + e^(-x))`

This makes it ideal for use before a loss function: it converts raw model outputs into probabilities, and its smoothness means gradients flow properly during backpropagation.

## 22. What is the difference between a loss function and a metric?

- **Loss**: used by the computer for automated learning via SGD. Must be smooth with meaningful gradients.
- **Metric**: used by humans to understand model performance. Doesn't need gradients. Examples: accuracy, precision, recall.

A metric reflects what we actually care about; a loss is a compromise - something similar enough to our goal but optimizable via gradient descent.

## 23. What is the function to calculate new weights using a learning rate?

`new_weight = old_weight - gradient * learning_rate`

In PyTorch: `param.data -= param.grad * lr`

We subtract because we want to minimize the loss: if gradient is positive (increasing the weight increases loss), we should decrease the weight.

## 24. What does the DataLoader class do?

`DataLoader` takes a Dataset and automatically:
- Splits it into mini-batches of a specified size
- Optionally shuffles the data each epoch for better generalization
- Returns batches as tuples of (inputs, targets) ready for training

This avoids manual batching and shuffling code.

## 25. Write pseudocode showing the basic steps taken in each epoch for SGD.

```
for each mini-batch (xb, yb) in training DataLoader:
    preds = model(xb)           # forward pass
    loss = loss_func(preds, yb)  # compute loss
    loss.backward()              # compute gradients
    for param in model.parameters():
        param.data -= param.grad * lr   # update
        param.grad = None               # reset
```

## 26. Create a function that returns `[(1,'a'),(2,'b'),(3,'c'),(4,'d')]` from `[1,2,3,4]` and `'abcd'`.

`list(zip([1,2,3,4], 'abcd'))`

The output is a Dataset: a list of (input, target) tuples. This is the standard data structure PyTorch expects, where you can index into it and get back a tuple of (x, y).

## 27. What does `view` do in PyTorch?

`view` reshapes a tensor without changing its underlying data. It just changes the interpretation of the same numbers in memory.

Example: `tensor.view(-1, 28*28)` takes images of shape `[N, 28, 28]` and flattens each to `[N, 784]`. The `-1` means "figure out this dimension automatically."

## 28. What are the "bias" parameters in a neural network? Why do we need them?

Bias is the `b` in `y = w*x + b`. Without bias, when all inputs are 0, the output is forced to be 0 (the line passes through the origin).

Bias gives the model flexibility to shift the output independently of the input - like the y-intercept in a line equation. It's essential for the model to fit real-world data.

## 29. What does the `@` operator do in Python?

Matrix multiplication. `A @ B` multiplies matrix A by matrix B.

In our model: `xb @ weights` computes `w1*x1 + w2*x2 + ... + w784*x784` for every image in the batch simultaneously. This is the core computation of a linear layer.

## 30. What does the `backward` method do?

`loss.backward()` triggers automatic differentiation: PyTorch traces back through all the operations that produced `loss`, computes the gradient of the loss with respect to every parameter that has `requires_grad=True`, and stores those gradients in each parameter's `.grad` attribute.

This is backpropagation - the backward pass through the network.

## 31. Why do we have to zero the gradients?

Because `loss.backward()` **adds** to the existing gradients rather than replacing them. If you don't zero them after each update, gradients accumulate across batches, giving incorrect updates.

So after each step: `param.grad = None` or `param.grad.zero_()`

## 32. What information do we have to pass to `Learner`?

```python
learn = Learner(
    dls,           # DataLoaders (training + validation)
    model,         # the neural network model
    opt_func,      # optimizer function (e.g. SGD)
    loss_func,     # loss function
    metrics        # metric(s) to track
)
```

The Learner ties together data, model, optimization, and evaluation into one training interface.

## 33. Show Python or pseudocode for the basic steps of a training loop.

```python
for epoch in range(num_epochs):
    for xb, yb in train_dl:
        preds = model(xb)
        loss = loss_func(preds, yb)
        loss.backward()
        opt.step()      # update parameters
        opt.zero_grad() # reset gradients
    # compute and print validation metrics
```

## 34. What is "ReLU"? Draw a plot of it for values from -2 to +2.

ReLU (Rectified Linear Unit) = replace every negative number with 0, leave positive numbers unchanged.

`relu(x) = max(0, x)`

Plot: a flat line at 0 for x < 0, then a 45-degree line going up for x >= 0. It looks like a hockey stick.

## 35. What is an "activation function"?

An activation function is a nonlinear function applied to the output of a linear layer. Without it, stacking multiple linear layers is mathematically equivalent to a single linear layer.

Activation functions break linearity, allowing the network to learn complex, nonlinear patterns. Common examples: ReLU, sigmoid, tanh.

## 36. What's the difference between `F.relu` and `nn.ReLU`?

- `F.relu` is a **function** from `torch.nn.functional`. You call it directly: `F.relu(x)`.
- `nn.ReLU` is a **module** (class). You instantiate it: `nn.ReLU()`, then call it like a function.

When building models with `nn.Sequential`, you must use module versions (`nn.ReLU()`). In standalone code, function versions (`F.relu(x)`) are fine. Both do exactly the same computation.

## 37. The universal approximation theorem shows that any function can be approximated as closely as needed using just one nonlinearity. So why do we normally use more?

The theorem is about theoretical possibility, not practical efficiency. Deeper models (more layers) achieve the same accuracy with **fewer total parameters** than wider shallow models.

Fewer parameters means: faster training, less memory, better generalization. Deeper networks can learn hierarchical features (edges -> shapes -> patterns -> objects) that generalize better.

In the 1990s, researchers over-relied on this theorem and rarely experimented with deep models - one reason for the "AI winter." Today, even 18-layer ResNet dramatically outperforms 2-layer models.